# Accuracy improvement experiment

This notebook compares stronger regularisation and different model families on the **original training partition only**. It keeps the original test evaluation frozen and never changes the application's active-model pointer.

Read top to bottom: load training records → compare candidates → check selection stability → save the best candidate → decide whether the evidence supports an improvement.

**Default:** display saved experimental results. To repeat the calculations, set `RUN_EXPERIMENT = True` and choose a new candidate run name. Training-only cross-validation is exploratory evidence, not a new independent test or clinical validation.

In [1]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

from pathlib import Path
from datetime import datetime, timezone
import json
import sys
import time

project = Path.cwd()
if not (project / "lung_tabular").exists():
    project = project.parent
if str(project) not in sys.path:
    sys.path.insert(0, str(project))

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import BernoulliNB, GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from lung_tabular import ARTIFACTS, DATA, FEATURES, LABELS, REPORTS, SCHEMA_VERSION, TARGET
from lung_tabular.data import read_prepared
from lung_tabular.model import load_model
from lung_tabular.utils import read_json, write_json, sha256

## 1 Settings

Use a new run name to preserve previously saved candidates. The model installed in the app is not replaced, even when the experiment is enabled.

In [2]:
RUN_EXPERIMENT = False
CANDIDATE_RUN_NAME = "accuracy_candidate_20261027"
REFERENCE_RUN = "numerical_v1"
RESULT_FILE = REPORTS / "accuracy_improvement.json"

if RUN_EXPERIMENT and (ARTIFACTS / CANDIDATE_RUN_NAME).exists():
    raise ValueError("This candidate run already exists. Choose a new name.")

## 2 Load training records and the reference model

Existing data hashes and split IDs are verified. Only rows marked train are supplied to model selection and validation.

In [3]:
records, summary = read_prepared(DATA, REPORTS)
training = records[records["split"] == "train"].reset_index(drop=True)
X = training.loc[:, list(FEATURES)]
y = training[TARGET]
incumbent, reference_metadata = load_model(ARTIFACTS / REFERENCE_RUN)
if reference_metadata["split_id"] != summary["split_id"]:
    raise ValueError("The reference model and training records use different splits.")
print("Training records:", len(training))
print("Reference run:", REFERENCE_RUN)

Training records: 2395
Reference run: numerical_v1


## 3 Define candidate models

Compare the reference model, a majority baseline, regularised linear models, feature interactions, nearest neighbours, tree ensembles and shallower boosting. Scaling stays inside each Pipeline so every fold fits its own preprocessing. No new libraries are needed.

In [4]:
def scaled(estimator):
    return Pipeline([('scale', StandardScaler()), ('classifier', estimator)])


models = {
    'current_hgb': clone(incumbent),
    'majority_baseline': scaled(DummyClassifier(strategy='most_frequent')),
    'logistic_strong_regularization': scaled(LogisticRegression(C=.001, max_iter=2000, random_state=42)),
    'logistic_regularization': scaled(LogisticRegression(C=.01, max_iter=2000, random_state=42)),
    'logistic_interactions': Pipeline([('pairs', PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)),
                                      ('scale', StandardScaler()), ('classifier', LogisticRegression(C=.01, max_iter=2000, random_state=42))]),
    'lda_shrinkage': scaled(LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')),
    'gaussian_nb': scaled(GaussianNB()),
    'bernoulli_nb': scaled(BernoulliNB()),
    'knn_51': scaled(KNeighborsClassifier(n_neighbors=51)),
    'knn_151': scaled(KNeighborsClassifier(n_neighbors=151)),
    'extra_trees_regularized': scaled(ExtraTreesClassifier(n_estimators=160, min_samples_leaf=15, n_jobs=1, random_state=42)),
    'random_forest_regularized': scaled(RandomForestClassifier(n_estimators=160, max_depth=4, min_samples_leaf=20, n_jobs=1, random_state=42)),
    'hgb_shallow': scaled(HistGradientBoostingClassifier(max_iter=100, max_leaf_nodes=3, learning_rate=.03,
                                                        l2_regularization=10, random_state=42)),
    'hgb_medium': scaled(HistGradientBoostingClassifier(max_iter=150, max_leaf_nodes=7, learning_rate=.03,
                                                       l2_regularization=10, random_state=42)),
    'hgb_deeper': scaled(HistGradientBoostingClassifier(max_iter=120, max_leaf_nodes=15, learning_rate=.05,
                                                       l2_regularization=10, random_state=42)),
}

## 4 Compare five fold training validation scores

Rank candidates by accuracy while retaining balanced accuracy, macro F1 and ROC AUC. The highest score after trying many candidates can be optimistic, so it is not sufficient for activating a replacement.

In [5]:
if RUN_EXPERIMENT:
    screening = []
    for name, model in models.items():
        started = time.perf_counter()
        scores = cross_validate(model, X, y, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
                                scoring=['accuracy', 'balanced_accuracy', 'f1_macro', 'roc_auc'], n_jobs=1, error_score='raise')
        row = {'model': name, 'seconds': time.perf_counter()-started}
        for metric in ['accuracy', 'balanced_accuracy', 'f1_macro', 'roc_auc']:
            row[metric] = float(scores['test_'+metric].mean())
            row[metric+'_folds'] = scores['test_'+metric].tolist()
        screening.append(row)
        print(name, f"accuracy={row['accuracy']:.4f}", f"balanced={row['balanced_accuracy']:.4f}",
              f"macroF1={row['f1_macro']:.4f}", flush=True)
else:
    screening = read_json(RESULT_FILE)["screening"]

screening_table = pd.DataFrame(screening).sort_values("accuracy", ascending=False)
display(screening_table[["model", "accuracy", "balanced_accuracy", "f1_macro", "roc_auc"]].round(4))

,model,accuracy,balanced_accuracy,f1_macro,roc_auc
12,hgb_shallow,0.5361,0.5359,0.5348,0.5344
13,hgb_medium,0.5307,0.5305,0.5301,0.5321
11,random_forest_regularized,0.5294,0.5286,0.5264,0.5305
0,current_hgb,0.5240,0.5242,0.5236,0.5309
14,hgb_deeper,0.5165,0.5163,0.5159,0.5264
9,knn_151,0.5161,0.5156,0.5150,0.5242
6,gaussian_nb,0.5098,0.5092,0.5079,0.5133
5,lda_shrinkage,0.5090,0.5083,0.5071,0.5103
10,extra_trees_regularized,0.5086,0.5082,0.5079,0.5112
3,logistic_regularization,0.5086,0.5079,0.5065,0.5111


## 5 Check model selection on different folds

On five outer training folds, use three inner folds to select among six shortlisted recipes. Compare its outer-fold predictions with a fresh clone of the reference model and the majority baseline. Also record the fixed shallow-boosting candidate as a sensitivity check.

The shortlist was informed by the screening above. All folds reuse a previously studied training dataset, so this is a stability diagnostic, not untouched independent evidence. The paired fold interval is descriptive because CV folds are correlated.

In [6]:
if RUN_EXPERIMENT:
    names = ['current_hgb', 'majority_baseline', 'hgb_shallow', 'hgb_medium', 'hgb_deeper', 'random_forest_regularized']
    outer_results = []
    for fold, (fit, validation) in enumerate(StratifiedKFold(5, shuffle=True, random_state=2026).split(X, y), 1):
        scores = []
        for name in names:
            cv = cross_validate(models[name], X.iloc[fit], y.iloc[fit],
                                cv=StratifiedKFold(3, shuffle=True, random_state=42),
                                scoring='accuracy', n_jobs=1, error_score='raise')
            scores.append((float(cv['test_score'].mean()), name))
        scores.sort(key=lambda row: (-row[0], row[1]))
        winner = scores[0][1]
        record = {'fold': fold, 'selected': winner, 'inner_accuracy': scores[0][0]}
        for label, name in [('selected', winner), ('current', 'current_hgb'), ('baseline', 'majority_baseline'),
                            ('shallow', 'hgb_shallow')]:
            model = clone(models[name]).fit(X.iloc[fit], y.iloc[fit])
            predicted = model.predict(X.iloc[validation])
            record[label+'_accuracy'] = float(accuracy_score(y.iloc[validation], predicted))
            record[label+'_balanced_accuracy'] = float(balanced_accuracy_score(y.iloc[validation], predicted))
            record[label+'_macro_f1'] = float(f1_score(y.iloc[validation], predicted, average='macro'))
        outer_results.append(record)
        print(json.dumps(record), flush=True)

    delta = np.array([row['selected_accuracy']-row['current_accuracy'] for row in outer_results])
    shallow_delta = np.array([row['shallow_accuracy']-row['current_accuracy'] for row in outer_results])
    nested_validation = {
        'partition': 'original training records only', 'records': len(training),
        'outer_folds': 5, 'outer_seed': 2026, 'inner_folds': 3, 'inner_seed': 42,
        'candidate_models': names, 'fold_results': outer_results,
        'selected_mean_accuracy': float(np.mean([row['selected_accuracy'] for row in outer_results])),
        'current_mean_accuracy': float(np.mean([row['current_accuracy'] for row in outer_results])),
        'baseline_mean_accuracy': float(np.mean([row['baseline_accuracy'] for row in outer_results])),
        'mean_improvement': float(delta.mean()), 'folds_improved': int((delta > 0).sum()),
        'paired_95_interval': [float(delta.mean()-2.776*delta.std(ddof=1)/np.sqrt(5)),
                               float(delta.mean()+2.776*delta.std(ddof=1)/np.sqrt(5))],
        'shallow_mean_accuracy': float(np.mean([row['shallow_accuracy'] for row in outer_results])),
        'shallow_mean_improvement': float(shallow_delta.mean()),
        'shallow_folds_improved': int((shallow_delta>0).sum()),
        'limitation': 'Training-only selection diagnostics on a previously studied dataset; not independent clinical evaluation.'
    }
else:
    nested_validation = read_json(RESULT_FILE)["nested_validation"]

display(pd.DataFrame(nested_validation["fold_results"])[[
    "fold", "selected", "selected_accuracy", "current_accuracy", "baseline_accuracy", "shallow_accuracy"
]].round(4))
print("Selection-process accuracy:", nested_validation["selected_mean_accuracy"])
print("Reference accuracy on the same outer folds:", nested_validation["current_mean_accuracy"])
print("Descriptive paired interval:", nested_validation["paired_95_interval"])

,fold,selected,selected_accuracy,current_accuracy,baseline_accuracy,shallow_accuracy
0,1,hgb_medium,0.5324,0.5261,0.5052,0.5574
1,2,current_hgb,0.5157,0.5157,0.5052,0.5407
2,3,hgb_medium,0.5282,0.5303,0.5052,0.5365
3,4,hgb_medium,0.4948,0.5177,0.5073,0.5094
4,5,hgb_shallow,0.5219,0.5386,0.5073,0.5219


Selection-process accuracy: 0.518580375782881
Reference accuracy on the same outer folds: 0.525678496868476
Descriptive paired interval: [-0.022277220530227786, 0.008080978359037807]


## 6 Save the best screened candidate separately

Fit the candidate on the training partition and save a complete compatible pipeline and metadata. No new test score is assigned. This creates an experimental model folder, not an active replacement.

In [7]:
if RUN_EXPERIMENT:
    started = time.perf_counter()
    best = max(screening, key=lambda row: row["accuracy"])
    candidate_name = best["model"]
    candidate = clone(models[candidate_name]).fit(X, y)
    candidate_run = ARTIFACTS / CANDIDATE_RUN_NAME
    candidate_run.mkdir(parents=True)
    joblib.dump(candidate, candidate_run / "model.joblib", compress=3)

    comparison = []
    for row in screening:
        scores = {"model": row["model"]}
        for old_name, new_name in [("accuracy", "accuracy"), ("balanced_accuracy", "balanced_accuracy"),
                                   ("f1_macro", "macro_f1"), ("roc_auc", "roc_auc")]:
            scores["cv_" + new_name] = row[old_name]
            scores["cv_" + new_name + "_std"] = float(np.std(row[old_name + "_folds"]))
        comparison.append(scores)
    comparison.sort(key=lambda row: (-row["cv_accuracy"], row["model"]))

    candidate_metadata = {
        "schema_version": SCHEMA_VERSION, "model": candidate_name, "run": CANDIDATE_RUN_NAME,
        "features": list(FEATURES), "labels": list(LABELS), "age_range": summary["age_range"],
        "split_id": summary["split_id"], "csv_sha256": summary["csv_sha256"],
        "model_sha256": sha256(candidate_run / "model.joblib"), "sklearn_version": sklearn.__version__,
        "training_records": len(training), "cv_folds": 5, "comparison": comparison,
        "selection_metric": "exploratory five-fold training accuracy; no automatic promotion",
        "training_seconds": time.perf_counter() - started,
        "created_utc": datetime.now(timezone.utc).isoformat(), "device": "CPU", "test_metrics": None,
        "classifier_parameters": candidate.named_steps["classifier"].get_params(),
        "quality_note": "Performance remains near chance; higher screened CV accuracy does not establish reliable improvement.",
        "evaluation_note": "CV-only experimental candidate. Original test results remain frozen. Not activated."
    }
    write_json(candidate_run / "metadata.json", candidate_metadata)
    pd.DataFrame(comparison).to_csv(candidate_run / "cv_comparison.csv", index=False)

    reloaded, _ = load_model(candidate_run)
    assert np.allclose(candidate.predict_proba(X.head()), reloaded.predict_proba(X.head()))
    print("Saved experimental candidate:", candidate_run)

## 7 Record the decision

Retain the current model unless the selection process improves by at least one percentage point on the outer folds and the descriptive paired interval is positive. This conservative rule is a gate for this experiment, not proof of clinical validity. No model is automatically activated by this notebook.

A higher screened score alone must not be described as improved test accuracy. Better verified data and independent evaluation are needed before making stronger performance claims.

In [8]:
if RUN_EXPERIMENT:
    supported = (nested_validation["mean_improvement"] >= 0.01
                 and nested_validation["paired_95_interval"][0] > 0)
    experiment = {
        "reference_run": REFERENCE_RUN, "candidate_run": CANDIDATE_RUN_NAME,
        "split_id": summary["split_id"], "partition": "original training records only",
        "selection_seed": 42, "screening": screening, "nested_validation": nested_validation,
        "improvement_supported": bool(supported), "activated": False,
        "decision": "Keep the current model; a reliable accuracy improvement has not been established."
            if not supported else "Review the candidate; no automatic activation.",
        "test_evaluation": "Original test report unchanged; no new held-out test score.",
        "limitation": "Model search on a previously studied dataset; validation gains may reflect selection noise."
    }
    write_json(RESULT_FILE, experiment)
    screening_table[["model", "accuracy", "balanced_accuracy", "f1_macro", "roc_auc"]].to_csv(
        REPORTS / "accuracy_screening.csv", index=False
    )
else:
    experiment = read_json(RESULT_FILE)

print(experiment["decision"])
print(experiment["test_evaluation"])
print("Candidate:", experiment["candidate_run"])
print("App model changed:", experiment["activated"])

Keep the current model; a reliable accuracy improvement has not been established.
Original test report unchanged; no new held-out test score.
Candidate: accuracy_candidate_20261027
App model changed: False
